In [1]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform

pd.set_option("display.float_format", "{:.2f}".format)

# OS에 따라 다른 폰트 지정
if platform.system() == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":  # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:  # Linux (예: Colab, Ubuntu)
    plt.rcParams["font.family"] = "NanumGothic"

In [2]:
portfolio = pd.read_csv("../dataset/portfolio.csv", index_col=0)
profile = pd.read_csv("../dataset/profile.csv", index_col=0)
transcript = pd.read_csv("../dataset/transcript.csv", index_col=0)
df_menu = pd.read_csv("../dataset/starbucks_menu_260112.csv", index_col=0)


for name, df in [("portfolio", portfolio), ("profile", profile), ("transcript", transcript)]:
    print(f"\n===== {name} =====")
    print(df.shape)
    print(df.dtypes)
    print(df.isnull().sum())
    print(df.head(3))


===== portfolio =====
(10, 6)
reward        int64
channels        str
difficulty    int64
duration      int64
offer_type      str
id              str
dtype: object
reward        0
channels      0
difficulty    0
duration      0
offer_type    0
id            0
dtype: int64
   reward                              channels  difficulty  duration  \
0      10         ['email', 'mobile', 'social']          10         7   
1      10  ['web', 'email', 'mobile', 'social']          10         5   
2       0            ['web', 'email', 'mobile']           0         4   

      offer_type                                id  
0           bogo  ae264e3637204a6fb9bb56bc8210ddfd  
1           bogo  4d5c57ea9a6940dd891ad53e9dbe8da0  
2  informational  3f207df678b143eea3cee63160fa8bed  

===== profile =====
(17000, 5)
gender                  str
age                   int64
id                      str
became_member_on      int64
income              float64
dtype: object
gender              2175
age       

# | **portfolio**

In [3]:
portfolio.isna().sum()

reward        0
channels      0
difficulty    0
duration      0
offer_type    0
id            0
dtype: int64

In [4]:
# channels 리스트가 아닌 문자열 모음이므로 변환 필요
print(portfolio["channels"])

0           ['email', 'mobile', 'social']
1    ['web', 'email', 'mobile', 'social']
2              ['web', 'email', 'mobile']
3              ['web', 'email', 'mobile']
4                        ['web', 'email']
5    ['web', 'email', 'mobile', 'social']
6    ['web', 'email', 'mobile', 'social']
7           ['email', 'mobile', 'social']
8    ['web', 'email', 'mobile', 'social']
9              ['web', 'email', 'mobile']
Name: channels, dtype: str


In [5]:
# channels: 문자열을 리스트로 변환
portfolio["channels"] = portfolio["channels"].apply(ast.literal_eval)

# 리스트 -> 이진 컬럼
# (리스트는 태블로에서 사용 불가)
for ch in ["web", "email", "mobile", "social"]:
    portfolio[f"ch_{ch}"] = portfolio["channels"].apply(lambda x: 1 if ch in x else 0)

# 파생변수 - 채널 수 (몇 개 채널로 발송됐는지)
# 채널을 많이 쓰거나 적게 쓸수록 어떤 변화가 있는지 관찰 위해
portfolio["channel_count"] = portfolio["channels"].apply(len)

print("✅ portfolio 전처리 완료")
print(portfolio[["offer_type", "ch_web", "ch_email", "ch_mobile", "ch_social", "channel_count"]])

✅ portfolio 전처리 완료
      offer_type  ch_web  ch_email  ch_mobile  ch_social  channel_count
0           bogo       0         1          1          1              3
1           bogo       1         1          1          1              4
2  informational       1         1          1          0              3
3           bogo       1         1          1          0              3
4       discount       1         1          0          0              2
5       discount       1         1          1          1              4
6       discount       1         1          1          1              4
7  informational       0         1          1          1              3
8           bogo       1         1          1          1              4
9       discount       1         1          1          0              3


# | **profile** 
- id, became_member_on 결측치 없음 확인 

In [6]:
# 결측치 확인
profile.isna().sum()

gender              2175
age                    0
id                     0
became_member_on       0
income              2175
dtype: int64

In [7]:
# age: 100세 초과는 결측치로 처리
profile["age"] = profile["age"].where(profile["age"] <= 100, other=None)

In [8]:
age_bins = [0, 29, 39, 49, 59, 69, 120]
age_labels = ["20대이하", "30대", "40대", "50대", "60대", "70대이상"]
profile["age_group"] = pd.cut(profile["age"], bins=age_bins, labels=age_labels, right=True)

print(profile["age_group"].value_counts(dropna=False))

age_group
50대      3541
60대      2991
70대이상    2879
40대      2309
NaN      2180
20대이하    1574
30대      1526
Name: count, dtype: int64


In [9]:
# gender: 결측치 'Unknown' 표시
profile["gender"] = profile["gender"].fillna("Unknown")

In [10]:
# became_member_on: datetime 으로 변환
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"].astype(str), format="%Y%m%d")

In [11]:
# 파생변수
profile["member_year"] = profile["became_member_on"].dt.year
profile["member_month"] = profile["became_member_on"].dt.month

In [12]:
profile["became_member_on"].describe()

count                         17000
mean     2017-02-23 13:12:10.164706
min             2013-07-29 00:00:00
25%             2016-05-26 00:00:00
50%             2017-08-02 00:00:00
75%             2017-12-30 00:00:00
max             2018-07-26 00:00:00
Name: became_member_on, dtype: object

In [13]:
base_date = profile["became_member_on"].max()
profile["tenure_days"] = (base_date - profile["became_member_on"]).dt.days

print(f"기준일: {base_date.date()}")
print(profile["tenure_days"].describe())

기준일: 2018-07-26
count   17000.00
mean      517.45
std       411.22
min         0.00
25%       208.00
50%       358.00
75%       791.00
max      1823.00
Name: tenure_days, dtype: float64


In [14]:
profile[["income"]].describe()

,income
count,14825.00
mean,65404.99
std,21598.30
min,30000.00
25%,49000.00
50%,64000.00
75%,80000.00
max,120000.00


In [15]:
# 소득 구간
income_bins = [0, 40000, 60000, 80000, 100000, 200000]
income_labels = ["~4만", "4~6만", "6~8만", "8~10만", "10만+"]
profile["income_group"] = pd.cut(profile["income"], bins=income_bins, labels=income_labels)

print(profile["income_group"].value_counts(dropna=False))
print("income 결측 수:", profile["income"].isna().sum())

income_group
6~8만     4567
4~6만     4558
8~10만    2559
NaN      2175
~4만      2135
10만+     1006
Name: count, dtype: int64
income 결측 수: 2175


In [16]:
print("\n가입연도 분포:")
print(profile["member_year"].value_counts())
print("\ngender 분포:")
print(profile["gender"].value_counts(dropna=False))


가입연도 분포:
member_year
2017    6469
2018    4198
2016    3526
2015    1830
2014     691
2013     286
Name: count, dtype: int64

gender 분포:
gender
M          8484
F          6129
Unknown    2175
O           212
Name: count, dtype: int64


In [17]:
print("\n✅ profile 전처리 완료")
print(profile[["gender", "age", "age_group", "income", "income_group", "member_year", "tenure_days"]].head(5))
print(profile.dtypes)


✅ profile 전처리 완료
    gender   age age_group    income income_group  member_year  tenure_days
0  Unknown   NaN       NaN       NaN          NaN         2017          529
1        F 55.00       50대 112000.00         10만+         2017          376
2  Unknown   NaN       NaN       NaN          NaN         2018           14
3        F 75.00     70대이상 100000.00        8~10만         2017          443
4  Unknown   NaN       NaN       NaN          NaN         2017          356
gender                         str
age                        float64
id                             str
became_member_on    datetime64[us]
income                     float64
age_group                 category
member_year                  int32
member_month                 int32
tenure_days                  int64
income_group              category
dtype: object


# | **transcript**

In [18]:
transcript.isna().sum()

person    0
event     0
value     0
time      0
dtype: int64

In [19]:
print(type(transcript["value"].iloc[0]))

<class 'str'>


In [ ]:
transcript["value_keys"] = transcript["value"].apply(lambda x: str(tuple(sorted(ast.literal_eval(x).keys()))))

pd.crosstab(transcript["event"], transcript["value_keys"])

value_keys,"('amount',)","('offer id',)","('offer_id', 'reward')"
event,,,
offer completed,0,0,33579
offer received,0,76277,0
offer viewed,0,57725,0
transaction,138953,0,0


- 크로스탭 결과를 보면 offer completed의 value key가 다른 이벤트와 다른 것을 알 수 있음.

In [21]:
# value_keys 컬럼 제거
transcript.drop(columns=["value_keys"], inplace=True)

In [22]:
transcript["value"].apply(lambda x: list(ast.literal_eval(x).keys())[0]).value_counts()

value
amount      138953
offer id    134002
offer_id     33579
Name: count, dtype: int64

In [ ]:
# value: 실제 자료형으로 변환 필요
# 주의: received/viewed → 'offer id' (공백)
#       completed → 'offer_id'  (언더스코어)  ← key 이름 다름!


def split_value(log):
    d = ast.literal_eval(log["value"])  # 문자열을 딕셔너리 자료형으로 변환

    if log["event"] == "transaction":
        return pd.Series({"amount": d.get("amount"), "offer_id": None, "reward_amount": None})

    elif log["event"] == "offer completed":
        return pd.Series(
            {"amount": None, "offer_id": d.get("offer_id"), "reward_amount": d.get("reward")}
        )  # 리워드 금액 추출

    else:
        return pd.Series({"amount": None, "offer_id": d.get("offer id"), "reward_amount": None})


# 모든 행에 함수(split_value) 적용
# value 컬럼 하나를 amount, offer_id, reward_amount 3개로 쪼개어 새 데이터프레임 생성
parsed = transcript.apply(split_value, axis=1)
# 원본 transcript 옆에 parsed 데이터프레임 붙이기
transcript = pd.concat([transcript, parsed], axis=1)
transcript.drop(columns=["value"], inplace=True)  # 원본 value컬럼 제거


# time -> day
# 시간 단위를 일단위로 바꾸기(시계열 분석 편의 위해)
transcript["day"] = transcript["time"] // 24

In [24]:
print(transcript["event"].value_counts())
print(transcript[["person", "event", "amount", "offer_id", "day"]].head(8))

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64
                             person           event  amount  \
0  78afa995795e4d85b5d9ceeca43f5fef  offer received     NaN   
1  a03223e636434f42ac4c3df47e8bac43  offer received     NaN   
2  e2127556f4f64592b11af22de27a7932  offer received     NaN   
3  8ec6ce2a7e7949b1bf142def7d0e0586  offer received     NaN   
4  68617ca6246f4fbc85e91a2a49552598  offer received     NaN   
5  389bc3fa690240e798340f5a15918d5c  offer received     NaN   
6  c4863c7985cf408faee930f111475da3  offer received     NaN   
7  2eeac8d8feae4a8cad5a6af0499a211d  offer received     NaN   

                           offer_id  day  
0  9b98b8c7a33c4b65b9aebfe6a799e6d9    0  
1  0b1e1539f2cc45b7b9fa7c272da2e1d7    0  
2  2906b810c7d4411798c6938adc9daaa5    0  
3  fafdcd668e3743c1bb461111dcafc2a4    0  
4  4d5c57ea9a6940dd891ad53e9dbe8da0    0  
5  f19421c1d4aa40978ebb69ca19b0e20d   

# | viewed 없이 completed만 있는 경우

- viewed 후 completed -> 캠페인이 행동을 유도한 것 <- 진짜 마케팅 효과
- viewed 없이 completed <- 캠페인 효과 아님